# 🚀 PLANO DE AÇÃO: MONITORAMENTO E PREDIÇÃO CLIMÁTICA (ARQUITETURA ZARR & ETL)

Este projeto estabelece uma infraestrutura de dados de alto desempenho para mapear as influências oceano-atmosféricas sobre o Leste do Nordeste Brasileiro (**ENEB**) e a **Região Metropolitana do Recife**. O foco é a criação de um dataset pronto para Machine Learning (ARCO - Analysis-Ready Cloud Optimized).

---

## 1. DOMÍNIOS GEOGRÁFICOS DE ANÁLISE (COORDENADAS NASA/WGS84)

Para garantir a precisão dos preditores e do alvo, definimos os seguintes recortes espaciais:

* **Domínio de Influência (Preditores - Atlântico Tropical):** * `Latitude: 30°N a 30°S` | `Longitude: 100°W a 20°E`
* **Domínio Alvo 01 (ENEB - Leste do Nordeste):** * `Latitude: 2°S a 12°S` | `Longitude: 34.5°W a 40°W`
* **Domínio Alvo 02 (Grande Recife - Escala Local):** * `Latitude: 7.50°S a 8.50°S` | `Longitude: 34.70°W a 35.50°W`

---

## 2. ARQUITETURA DO PIPELINE (ETL INDUSTRIAL)
O processamento é executado em um **Loop Mestre Anual (1990–2025)** utilizando a técnica de **Janela Deslizante** para otimização de storage.

### Módulo A: Extração e Auditoria de Cache
* **ERA5 (Atmosfera):** Download de 15 variáveis. Todas recebem o sufixo `-ERA5` para garantir rastreabilidade total.
* **ORAS5 (Oceano):** Download de 5 variáveis críticas (SST, Camada Misturada, Calor Oceânico, Salinidade e SSH).
* **Auditoria Automática:** O motor de download valida a integridade de cada arquivo `.nc` (NetCDF). Caso detecte corrupção (HDF Error), o sistema deleta e reinicia o download automaticamente.

### Módulo B: Transformação e Resampling
* **Tratamento de Fluxos:** A Precipitação Total é processada via **soma acumulada diária**, enquanto as demais variáveis são convertidas em **médias diárias**.
* **Correção de Grade:** Tratamento especial para a grade curvilínea do ORAS5, garantindo que o cálculo de interpolação temporal ocorra antes da limpeza de coordenadas, evitando arquivos vazios (0GB).

### Módulo C: Consolidação Zarr e Self-Cleaning
* **Escrita Otimizada:** Os dados são salvos no formato **Zarr**, permitindo acesso paralelo e fatiado (*chunked*).
* **Janela Deslizante:** O cache de arquivos brutos (~7.5 GB/ano) é deletado imediatamente após a validação do arquivo Zarr final, mantendo o uso de disco constante e baixo.

---

## 3. VARIÁVEIS MONITORADAS (TOTAL: 20)

| Fonte | Categoria | Variáveis Principais (Assinatura Individual) |
| :--- | :--- | :--- |
| **ERA5** | **Superfície (SL)** | Precipitação, Temperatura-2m, Pressão MSL, Ventos-10m, CAPE, Fluxos de Calor, Água Precipitável (TCWV). |
| **ERA5** | **Altitude (PL)** | Geopotencial, Umidade Específica e Ômega (Velocidade Vertical) em 850, 500 e 200 hPa. |
| **ORAS5** | **Oceano (Deep)** | TSM (SST), Salinidade, Altura do Mar (SSH), Camada Misturada (MLD) e Conteúdo de Calor (OHC). |

---

## 4. OBJETIVOS E INTELIGÊNCIA ARTIFICIAL
Os dados consolidados alimentam os seguintes estágios de pesquisa:

1.  **Mapeamento de Feature Importance:** Uso de **XGBoost** e **Random Forest** para identificar quais coordenadas geográficas do Atlântico têm maior poder preditivo sobre as chuvas no Recife.
2.  **Análise Explicativa (SHAP):** Aplicação de valores SHAP para ranquear se a dinâmica atmosférica (Ventos/Pressão) ou a termodinâmica oceânica (Calor/TSM) é o driver principal de eventos extremos.
3.  **Estudo de Defasagem (Lag):** Identificação do tempo de resposta entre anomalias oceânicas remotas e a resposta pluviométrica local (lags de 1 a 6 meses).

In [ ]:
import os
import cdsapi
import xarray as xr
import pandas as pd
import time
import zipfile

# MÓDULO 1: CONFIGURAÇÕES E CLIENTE
# ------------------------------------------------------------------------------
CDS_URL = "https://cds.climate.copernicus.eu/api"
CDS_KEY = "e070e6f5-c003-4a24-8d53-c40d433402cd"
c = cdsapi.Client(url=CDS_URL, key=CDS_KEY)

BASE_DIR = os.getcwd()
CACHE_DIR = os.path.join(BASE_DIR, "cache_nc")
DATA_DIR = os.path.join(BASE_DIR, "data_zarr")
for d in [CACHE_DIR, DATA_DIR]: os.makedirs(d, exist_ok=True)

RANGE_ANOS = range(1990, 2026) 
AREA = [30, -100, -30, 20] 

# MÓDULO 2: DICIONÁRIOS COM NOMES EXTENSOS E ASSINATURA -ERA5
# ------------------------------------------------------------------------------
VARS_ERA5_SL = {
    'total_precipitation': 'Total-Precipitation-ERA5',
    '2m_temperature': 'Temperature-At-2-Meters-ERA5',
    'sea_surface_temperature': 'Sea-Surface-Temperature-ERA5', 
    'mean_sea_level_pressure': 'Mean-Sea-Level-Pressure-ERA5',
    '10m_u_component_of_wind': 'Zonal-Wind-At-10-Meters-ERA5',
    '10m_v_component_of_wind': 'Meridional-Wind-At-10-Meters-ERA5',
    'convective_available_potential_energy': 'Convective-Available-Potential-Energy-ERA5',
    'surface_latent_heat_flux': 'Surface-Latent-Heat-Flux-ERA5',
    'surface_sensible_heat_flux': 'Surface-Sensible-Heat-Flux-ERA5',
    'total_column_water_vapour': 'Total-Column-Water-Vapour-ERA5'
}

VARS_ERA5_PL = {
    'u_component_of_wind': 'Zonal-Wind-Altitude-ERA5',
    'v_component_of_wind': 'Meridional-Wind-Altitude-ERA5',
    'specific_humidity': 'Specific-Humidity-Altitude-ERA5',
    'geopotential': 'Geopotential-Height-ERA5',
    'vertical_velocity': 'Vertical-Velocity-Omega-ERA5'
}

VARS_ORAS5 = {
    'sea_surface_temperature': 'Sea-Surface-Temperature-ORAS5',
    'mixed_layer_depth_0_01': 'Mixed-Layer-Depth-ORAS5',
    'ocean_heat_content_for_the_upper_300m': 'Ocean-Heat-Content-300-Meters-ORAS5',
    'salinity': 'Sea-Water-Salinity-ORAS5', 
    'sea_surface_height': 'Sea-Surface-Height-ORAS5'
}

# MÓDULO 3: MOTOR DE DOWNLOAD (VALIDAÇÃO DE INTEGRIDADE)
# ------------------------------------------------------------------------------
def download_engine(dataset, var_key, source, full_name, ano_ref):
    target_nc = os.path.join(CACHE_DIR, f"{source}_{full_name}_{ano_ref}.nc")
    target_zip = target_nc.replace(".nc", ".zip")
    
    if os.path.exists(target_nc):
        try:
            with xr.open_dataset(target_nc, engine='netcdf4') as tmp: return target_nc
        except Exception:
            pass
        try: os.remove(target_nc)
        except: pass

    request = {'variable': [var_key], 'year': [str(ano_ref)], 'month': [f"{m:02d}" for m in range(1, 13)], 'format': 'netcdf'}
    if 'oras5' in dataset:
        request.update({'product_type': ['consolidated'], 
                        'vertical_resolution': ['all_levels' if var_key == 'salinity' else 'single_level']})
        final_path = target_zip 
    else:
        request.update({'product_type': 'reanalysis', 'time': ['00:00', '06:00', '12:00', '18:00'], 
                        'area': AREA, 'day': [f"{d:02d}" for d in range(1, 32)]})
        if 'pressure-levels' in dataset: request.update({'pressure_level': ['850', '500', '200']})
        final_path = target_nc

    try:
        print(f"[*] Solicitando: {full_name} ({ano_ref})")
        c.retrieve(dataset, request).download(final_path)
        if final_path.endswith('.zip'):
            with zipfile.ZipFile(final_path, 'r') as z:
                ext_name = z.namelist()[0]
                z.extract(ext_name, CACHE_DIR)
                os.rename(os.path.join(CACHE_DIR, ext_name), target_nc)
            if os.path.exists(final_path): os.remove(final_path)
        return target_nc
    except Exception as e:
        print(f"❌ Erro em {full_name}: {e}"); return None

# MÓDULO 4: UNIFICAÇÃO INDEPENDENTE (ZARR CONSOLIDADO)
# ------------------------------------------------------------------------------
def create_zarr_process(files, name, ano_ref, is_oras=False):
    if not files: return None
    ds_list = []
    print(f"⚙️ Processando {name} Zarr {ano_ref}...")
    for f in files:
        ds = xr.open_dataset(f)
        if is_oras:
            t_coord = 'time_counter' if 'time_counter' in ds.coords else 'time'
            vars_to_keep = list(ds.data_vars)
            ds_d = ds[vars_to_keep].resample({t_coord: '1D'}).mean()
            if t_coord != 'time': ds_d = ds_d.rename({t_coord: 'time'})
            ds_d = ds_d.drop_vars(['nav_lat', 'nav_lon', 'deptht'], errors='ignore')
        else:
            ds = ds.drop_vars(['expver', 'number'], errors='ignore')
            if 'valid_time' in ds.coords: ds = ds.rename({'valid_time': 'time'})
            ds_d = ds.resample(time='1D').sum() if 'Precipitation' in f else ds.resample(time='1D').mean()
        
        ds_list.append(ds_d.compute())
        ds.close()

    output_path = os.path.join(DATA_DIR, f"{name}_Zarr_{ano_ref}.zarr")
    xr.merge(ds_list, compat='override').to_zarr(output_path, mode='w', consolidated=True)
    return output_path

# MÓDULO 5: AUDITORIA, RELATÓRIO TÉCNICO E LIMPEZA (SELF-CLEANING)
# ------------------------------------------------------------------------------
def get_size_gb(path):
    if not os.path.exists(path): return 0
    if os.path.isfile(path): return os.path.getsize(path) / (1024**3)
    total = 0
    for dp, dn, filenames in os.walk(path):
        for f in filenames: total += os.path.getsize(os.path.join(dp, f))
    return total / (1024**3)

def auditoria_ciclo(ano_ref, files_nc, paths_zarr):
    print(f"📊 Relatório e Auditoria - Ano {ano_ref}")
    size_zarr = sum(get_size_gb(p) for p in paths_zarr)
    
    # Critério de Auditoria: Arquivos Zarr devem existir e ter tamanho razoável
    if size_zarr > 0.1: # Exemplo: maior que 100MB
        print(f"✅ Auditoria aprovada ({size_zarr:.4f} GB). Limpando cache...")
        for f in files_nc:
            try:
                if os.path.exists(f): os.remove(f)
            except: pass
        return True
    else:
        print(f"❌ Falha na auditoria do ano {ano_ref}. Cache preservado.")
        return False

# MÓDULO 6: EXECUÇÃO DO PIPELINE (LOOP MESTRE)
# ------------------------------------------------------------------------------
global_start = time.time()

for ano in RANGE_ANOS:
    print(f"\n{'='*80}\n📅 INICIANDO PROCESSAMENTO DO ANO: {ano}\n{'='*80}")
    
    # 1. Verificação de existência (Skip if exists)
    path_e_final = os.path.join(DATA_DIR, f"ERA5_Zarr_{ano}.zarr")
    path_o_final = os.path.join(DATA_DIR, f"ORAS5_Zarr_{ano}.zarr")
    if os.path.exists(path_e_final) and os.path.exists(path_o_final):
        print(f"⏭️  Ano {ano} já concluído. Pulando para o próximo...")
        continue

    # 2. Filas de Download
    lista_era5, lista_oras5 = [], []
    for dset, dic, src in [('reanalysis-era5-single-levels', VARS_ERA5_SL, 'ERA5'),
                           ('reanalysis-era5-pressure-levels', VARS_ERA5_PL, 'ERA5')]:
        for k, v in dic.items():
            res = download_engine(dset, k, src, v, ano)
            if res: lista_era5.append(res)

    for k, v in VARS_ORAS5.items():
        res = download_engine('reanalysis-oras5', k, 'ORAS5', v, ano)
        if res: lista_oras5.append(res)

    # 3. Processamento para Zarr
    p_e = create_zarr_process(lista_era5, "ERA5", ano)
    p_o = create_zarr_process(lista_oras5, "ORAS5", ano, is_oras=True)

    # 4. Auditoria e Limpeza de Janela Deslizante
    if not auditoria_ciclo(ano, lista_era5 + lista_oras5, [p_e, p_o]):
        print("停止 - O pipeline parou para evitar perda de dados.")
        break

print(f"\n{'='*80}\n🏁 PIPELINE FINALIZADO COM SUCESSO\nTempo Total: {(time.time()-global_start)/3600:.2f}h\n{'='*80}")


📅 INICIANDO PROCESSAMENTO DO ANO: 1990
[*] Solicitando: Total-Precipitation-ERA5 (1990)


2026-03-16 14:22:09,566 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-16 14:22:09,566 INFO Request ID is 7ee27afe-be95-467b-9c1d-25fec3033732
2026-03-16 14:22:09,790 INFO status has been updated to accepted
2026-03-16 14:23:26,753 INFO status has been updated to successful


[*] Solicitando: Temperature-At-2-Meters-ERA5 (1990)


2026-03-16 14:24:20,734 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-16 14:24:20,736 INFO Request ID is 5d9be91a-d49f-420a-821a-be89a788940a
2026-03-16 14:24:20,967 INFO status has been updated to accepted
2026-03-16 14:24:35,283 INFO status has been updated to successful


[*] Solicitando: Sea-Surface-Temperature-ERA5 (1990)


...

2026-03-16 14:52:22,591 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-16 14:52:22,592 INFO Request ID is 21539dd1-d657-4393-9fc0-2a6ee4407742
2026-03-16 14:52:22,824 INFO status has been updated to accepted
2026-03-16 14:52:37,242 INFO status has been updated to running
2026-03-16 14:55:17,793 INFO status has been updated to successful


[*] Solicitando: Convective-Available-Potential-Energy-ERA5 (1990)


2026-03-16 15:11:14,783 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-16 15:11:14,786 INFO Request ID is 8164dcc6-90aa-483a-915e-56833bc21d98
2026-03-16 15:11:15,117 INFO status has been updated to accepted
2026-03-16 15:11:27,774 INFO status has been updated to running
2026-03-16 15:15:43,051 INFO status has been updated to successful


[*] Solicitando: Surface-Latent-Heat-Flux-ERA5 (1990)


2026-03-16 15:17:38,136 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-16 15:17:38,137 INFO Request ID is 82223674-321c-47d2-a5f7-ea6ed0ecb0d8
2026-03-16 15:17:38,485 INFO status has been updated to accepted
2026-03-16 15:17:48,804 INFO status has been updated to running
2026-03-16 15:22:05,987 INFO status has been updated to successful
faea24d0d24db6b1300bac6f34209035.nc:  28%|██▊       | 92.0M/323M [00:39<02:14, 1.81MB/s]